# Backpropagation

Loss tells us the model is wrong. **Backpropagation** calculates how each weight and bias contributed to that loss.

```mermaid
flowchart LR
    F[Features] -->|Forward pass| M[Model]
    M --> S[Scores]
    S --> L[Loss]
    T[Targets] --> L
    L -->|Backward pass| G[Gradients]
    G --> U[Update parameters]
    U --> M
```

## Derivatives and Gradients

Think of the loss as a valley:

- Steep slope = large gradient
- Gentle slope = small gradient
- Flat bottom = gradient near zero

The update moves in the opposite direction of the gradient.

```text
new parameter = old parameter - learning rate * gradient
```

A convex function has one bowl-shaped global minimum. Neural-network loss is usually non-convex and can have many valleys, so gradient descent searches for a low-loss point.

In [8]:
import torch
from torch import nn

torch.manual_seed(42)

# 5 animals, 6 features each
input_data = torch.tensor([
    [-0.4421,  1.5207,  2.0607, -0.3647,  0.4691,  0.0940],
    [-0.9155, -0.0475, -1.3645,  0.6336, -1.9520, -0.3398],
    [ 0.7406,  1.6763, -0.8511,  0.2432,  0.1123, -0.0633],
    [-1.6630, -0.0718, -0.1285,  0.5396, -0.0288, -0.8622],
    [-0.7413,  1.7920, -0.0883, -0.6685,  0.4745, -0.4245],
])

targets = torch.tensor([0, 0, 2, 1, 1])

model = nn.Sequential(
    nn.Linear(6, 8),   # model[0]
    nn.ReLU(),        # model[1]
    nn.Linear(8, 4),   # model[2]
    nn.ReLU(),        # model[3]
    nn.Linear(4, 3),   # model[4]
)

criterion = nn.CrossEntropyLoss()

## 1. Forward Pass and Loss

The model returns three raw scores for each animal. The loss compares them with the target class numbers.

In [9]:
scores = model(input_data)
loss = criterion(scores, targets)

print(f"Scores shape: {scores.shape}")
print(f"Loss: {loss.item():.4f}")

Scores shape: torch.Size([5, 3])
Loss: 1.0748


## 2. Backward Pass

`loss.backward()` works backward through the network and stores each parameter's gradient in `.grad`.

```mermaid
flowchart RL
    L[Loss] --> O[Output layer]
    O --> H2[Hidden layer 2]
    H2 --> H1[Hidden layer 1]
```

In [10]:
# before backward
print(f"Gradient of model[0].weight before backward: {model[0].weight.grad}")
print(model[0].weight)

Gradient of model[0].weight before backward: None
Parameter containing:
tensor([[ 0.3121,  0.3388, -0.0956,  0.3750, -0.0894,  0.0824],
        [-0.1988,  0.2398,  0.3599, -0.2995,  0.3548,  0.0764],
        [ 0.3016,  0.0553,  0.1969, -0.0576,  0.3147,  0.0603],
        [-0.1906,  0.1041, -0.1881, -0.0479, -0.1658,  0.2708],
        [-0.3223, -0.1882, -0.1153, -0.2455,  0.0385, -0.4032],
        [ 0.3687, -0.3468,  0.3152,  0.0679, -0.1326,  0.2523],
        [ 0.0636,  0.3298,  0.0446, -0.1288,  0.1097, -0.1107],
        [ 0.1718,  0.3645,  0.2360, -0.1785,  0.2357,  0.0730]],
       requires_grad=True)


In [11]:
loss.backward()
print(f"Gradient of model[0].weight after backward: {model[0].weight.grad}")
print(model[0].weight)

Gradient of model[0].weight after backward: tensor([[-0.0166, -0.0042,  0.0120, -0.0035, -0.0067, -0.0019],
        [ 0.0034, -0.0104, -0.0100,  0.0029, -0.0031,  0.0002],
        [ 0.0093, -0.0318, -0.0431,  0.0076, -0.0098, -0.0020],
        [ 0.0065, -0.0185, -0.0008,  0.0076, -0.0073,  0.0039],
        [ 0.0098,  0.0004, -0.0023, -0.0024, -0.0044,  0.0054],
        [-0.0054,  0.0184,  0.0250, -0.0044,  0.0057,  0.0011],
        [-0.0520,  0.0126, -0.0209,  0.0032,  0.0035, -0.0293],
        [-0.0112, -0.0020,  0.0138, -0.0063,  0.0026, -0.0010]])
Parameter containing:
tensor([[ 0.3121,  0.3388, -0.0956,  0.3750, -0.0894,  0.0824],
        [-0.1988,  0.2398,  0.3599, -0.2995,  0.3548,  0.0764],
        [ 0.3016,  0.0553,  0.1969, -0.0576,  0.3147,  0.0603],
        [-0.1906,  0.1041, -0.1881, -0.0479, -0.1658,  0.2708],
        [-0.3223, -0.1882, -0.1153, -0.2455,  0.0385, -0.4032],
        [ 0.3687, -0.3468,  0.3152,  0.0679, -0.1326,  0.2523],
        [ 0.0636,  0.3298,  0.0446, -

In [13]:
for name, parameter in model.named_parameters():
    print(
        f"{name:8} | parameter: {list(parameter.shape)} "
        f"| gradient: {list(parameter.grad.shape)}"
    )

0.weight | parameter: [8, 6] | gradient: [8, 6]
0.bias   | parameter: [8] | gradient: [8]
2.weight | parameter: [4, 8] | gradient: [4, 8]
2.bias   | parameter: [4] | gradient: [4]
4.weight | parameter: [3, 4] | gradient: [3, 4]
4.bias   | parameter: [3] | gradient: [3]


You can also access one layer directly:

```python
model[0].weight       # first layer weights
model[0].weight.grad  # first layer weight gradients
model[0].bias.grad    # first layer bias gradients
```

In [ ]:
model[0].weight
model[0].bias

# There are 6 inputs per row and each has their own weight

Parameter containing:
tensor([ 0.2073, -0.2488, -0.4041, -0.1577, -0.3131,  0.3350,  0.1176,  0.1691],
       requires_grad=True)

## 3. Manual Parameter Update

The learning rate controls the size of the update. Parameter updates should not become part of the gradient calculation, so use `torch.no_grad()`.

In [14]:
learning_rate = 0.01
loss_before_update = loss.item()

with torch.no_grad():
    for parameter in model.parameters():
        parameter -= learning_rate * parameter.grad

model.zero_grad()
loss_after_update = criterion(model(input_data), targets).item()

print(f"Before update: {loss_before_update:.4f}")
print(f"After update:  {loss_after_update:.4f}")

Before update: 1.0748
After update:  1.0743


## 4. Gradient Descent with SGD

An optimizer updates all parameters for us. `SGD` means stochastic gradient descent.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

optimizer.zero_grad()
scores = model(input_data)
loss = criterion(scores, targets)
loss.backward()  # gets your gradients
optimizer.step()  # calculates the new weights and biases weight = weight - lr * grad

with torch.no_grad():
    new_loss = criterion(model(input_data), targets)

print(f"Before SGD step: {loss.item():.4f}")
print(f"After SGD step:  {new_loss.item():.4f}")

Before SGD step: 1.0743
After SGD step:  1.0695


## Quick Look

```python
optimizer.zero_grad()       # clear old gradients
scores = model(input_data)  # forward pass
loss = criterion(scores, targets)
loss.backward()             # calculate gradients
optimizer.step()            # update weights and biases
```

> Gradients accumulate in PyTorch, so clear them before the next backward pass.